# 11 - Context engineering: budgets, isolation and compression

Large contexts cost money, add latency and dilute attention. This system keeps every model's context small:

1. **Token budgets** - `build_context` keeps the most relevant chunks within a budget, then orders them for reading.
2. **Context isolation** - each specialist works in its own context (tool outputs, skill files, scratch notes); the
   Coordinator only sees the compact `AgentReport`.
3. **Compression** - the Coordinator works from digests (findings + evidence IDs), a short conversation window and
   distilled long-term facts; Deep Agents also summarise long specialist histories automatically.

**Prerequisites:** `.env` with `OPENAI_API_KEY`, and the vector stores (`make ingest`).

In [ ]:
# Setup: make src/ importable and load the settings from .env
import sys, warnings
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore", message=".*langchain.mcp.*")

from agentic_rag.config import AgentSettings

settings = AgentSettings.load(ROOT)
print(f"Coordinator: {settings.model} ({settings.coordinator_effort}) | specialists: {settings.specialist_model} "
      f"({settings.specialist_effort}) | judge: {settings.judge_model} (pass >= {settings.judge_pass_score})")

## 1. Token budgets

In [ ]:
from agentic_rag.rag import build_context
from agentic_rag.retrieval import hybrid_search

chunks = hybrid_search(settings, "hr", "travel per diem hotel limits and mileage rates", k=12)
for budget in (600, 1500, 4000):
    pack = build_context(chunks, max_tokens=budget)
    print(f"budget {budget:>5}: kept {len(pack.sources):>2} chunks, {pack.tokens:>5} tokens, dropped {pack.dropped}")
print()
print(build_context(chunks, max_tokens=600).text[:800])

## 2. Context isolation: what a specialist reads vs what the Coordinator receives

In [ ]:
from agentic_rag.agents import hr_policy
from agentic_rag.agents.base import build_specialist, task_message
from agentic_rag.agents.reports import AgentReportDraft
from data_ingestion.metadata import token_encoder
from langchain_core.messages import HumanMessage

enc = token_encoder("gpt-4o")
agent = build_specialist(hr_policy.SPEC, settings, hr_policy.tools(settings))
goal = "What are the per diem rates for domestic travel by city tier, and the car mileage rate?"
state = await agent.ainvoke({"messages": [HumanMessage(task_message(goal, "s1", goal))]})

specialist_tokens = sum(len(enc.encode(str(m.content))) for m in state["messages"])
report = state["structured_response"]
report_tokens = len(enc.encode(report.model_dump_json()))
print(f"Specialist context: {len(state['messages'])} messages, ~{specialist_tokens} tokens (tool outputs, skills, reasoning)")
print(f"Report to Coordinator: ~{report_tokens} tokens ({report_tokens / max(specialist_tokens, 1):.0%})")
print()
print(report.findings)

In [ ]:
# Which messages filled the specialist's context?
for m in state["messages"]:
    calls = [c["name"] for c in getattr(m, "tool_calls", [])]
    print(f"{m.type:<6} {len(enc.encode(str(m.content))):>6} tokens {('calls ' + str(calls)) if calls else getattr(m, 'name', '') or ''}")

## 3. What the Coordinator's prompts contain
`reports_digest` is the compressed view used for reflection and synthesis; the conversation window keeps the last six messages, each truncated; long-term memory stores distilled one-line facts instead of transcripts.

In [ ]:
from agentic_rag.agents.reports import AgentReport
from agentic_rag.graph.nodes import Coordinator

fake_state = {"reports": [AgentReport(**report.model_dump(), agent="hr_policy", step_id="s1", goal=goal).model_dump()]}
print(Coordinator.reports_digest(fake_state))
print()
print(Coordinator.evidence_text(fake_state)[:1200])

## What to observe
- Halving the budget drops the *least relevant* chunks first; the rest stay in reading order.
- The specialist may read thousands of tokens of tool output; the Coordinator gets a few hundred.
- Every Coordinator prompt is built from digests, never from raw tool output.